## Extinction, galactic latitude and peak absolute magnitude

### To run this notebook, please [follow the instructions](https://lasair-lsst.readthedocs.io/en/main/core_functions/python-notebooks.html) or else it won`t work.
The instructions are at https://lasair-lsst.readthedocs.io/en/main/core_functions/python-notebooks.html

This notebook computes three Lasair features: extinction E(B-V), galactic latitude, 
and peak extinction-corrected absolute magnitude (PECAM).

In [9]:
import os, sys
sys.path.append('..')
import math, json, redshift_distance

### 1. Extinction
#### Get the dustmaps package set up. 
We use this package to compute the extinction from the dustmap of 
[Schlegel, Finkbeiner, and Davis](https://iopscience.iop.org/article/10.1086/305772).
Once the package is installed, the dustmaps themselves must be downloaded with fetch() -- change the
directory name according to your own setup.
Note that the fetch() method only needs to be called once. 

In [10]:
# Uncomment this to install the dustmaps package
#!/usr/bin/pip3 install dustmaps
from dustmaps.config import config

# Set this to a location on your own system where you want the two 64 Mbyte dustmap files
config['data_dir'] = '/Users/rwillia5/Desktop'
import dustmaps.sfd

# Uncomment the following to download the SFD dustmap
# dustmaps.sfd.fetch()

from dustmaps.sfd import SFDQuery
from astropy.coordinates import SkyCoord
sfd = SFDQuery()

#### Corrected colour for extinction
These colour corrections are multiplied by the extinction E(B-V) to get a magnitude correction. 
They arefrom Table 6 of 
[Schlafly and Finkbeiner](https://iopscience.iop.org/article/10.1088/0004-637X/737/2/103) with RV=3.1

In [11]:
# The LSST bands
EXTCOEF   = {'u':4.145, 'g':3.237, 'r':2.273, 'i':1.684, 'z':1.323, 'y':1.088}

# Modify magnitude for extinction as
# mag - ebv*EXTCOEF[band]

#### Extinction from sky position

In [12]:
def computeExtinction(ra, decl):
    c = SkyCoord(ra, decl, unit="deg", frame='icrs')
    ebv = float(sfd(c))
    return ebv

### 2. Magnitude and Flux
#### Transform between magnitudes and nanoJanskies

In [13]:
def mag2flux(mag):
    # flux in nanoJ
    flux =  math.pow(10, (31.4 - mag)/2.5)
    return flux
def flux2mag(flux):
    # flux in nanoJ
    mag = 31.4 - 2.5*math.log10(flux)
    return mag

### 3. Peak extinction-corrected magnitude
From the E(B-V) and lightcurve in flux, we compute magnitude, 
correct them with extinction, then find the brightest point, and the associated band.

In [14]:
def findPeakExtMag(ebv, z, lightcurve):
    peakMag  = 100
    peakBand = None
    for diaSource in lightcurve:
        mjd        = diaSource['midpointMjdTai']
        band       = diaSource['band']
        psfFlux    = diaSource['psfFlux']
        psfFluxErr = diaSource['psfFluxErr']
        if psfFlux > 0:
            mag = flux2mag(psfFlux)
            # extinction correction and k-correction
            correctedMag = mag - ebv*EXTCOEF[band] + 2.5*math.log(1+z)
            if correctedMag < peakMag:
                peakMag = correctedMag
                peakBand = band
    return (peakMag, peakBand)

### 4. Galactic latitude

In [15]:
# https://en.wikipedia.org/wiki/Galactic_coordinate_system
def galacticLat(ra, decl):
    alphaNGP = 192.85948
    deltaNGP =  27.1283
    sdngp = math.sin(math.radians(deltaNGP))
    cdngp = math.cos(math.radians(deltaNGP))
    sde = math.sin(math.radians(decl))
    cde = math.cos(math.radians(decl))
    cra = math.cos(math.radians(ra - alphaNGP))
    glat = math.degrees(math.asin(sdngp*sde + cdngp*cde*cra))
    return glat

### 5. Test run
Use the Lasair API to find some objects with a host galaxy, then compute their features.
Make sure to connect to the right endpoint.

In [16]:
#!/usr/bin/pip3 install lasair
from lasair import LasairError, lasair_client as lasair

API_TOKEN = os.getenv('LASAIR_LSST_TOKEN')
if API_TOKEN is None:
    print("No Token found. Check Spelling. Note that if you have just added your token to your environment variables, you may need to restart your terminal so your shell settings are reloaded. ")
    
endpoint = "https://api.lasair.lsst.ac.uk/api"
L = lasair(API_TOKEN, endpoint=endpoint)


#### Get some working objects with a Sherlock redshift

In [17]:
selected = """ 
  objects.diaObjectId, objects.ra, objects.decl, 
  sherlock_classifications.z, sherlock_classifications.photoz
"""

tables = 'objects,sherlock_classifications'

conditions = """
  objects.nDiaSources > 5 AND
  classification="SN" AND 
  (sherlock_classifications.z > 0 OR sherlock_classifications.photoz > 0)
"""

results = L.query(selected, tables, conditions, limit = 10)
for row in results:

    objectId = row['diaObjectId']
    ra       = row['ra']
    decl     = row['decl']
    z        = row['z']
    photoz   = row['photoz']
    if not z:
        z = photoz
    obj   = L.object(objectId, lasair_added=False, lite=True)

    # compute galactic latitude
    ebv = computeExtinction(ra, decl)
    
    # compute extinction
    galLat = galacticLat(ra, decl)

    # compute peak extinction-corrected apparent magnitude
    (peakMag, peakBand) = findPeakExtMag(ebv, z, obj['diaSourcesList'])
    if not peakBand:
        continue

    # combine z and apparent mag to get absolute mag
    # using Ken Smith code from Atlas for distance modulus
    distances = redshift_distance.redshiftToDistance(z)
    distanceModulus = distances['dmod']

    absMag = peakMag - distanceModulus

    print('%s: ra=%.2f decl=%.2f z=%.3f E(B-V)=%.2f peakMax=%.2f peakBand=%s, absMag=%.2f' % 
          (objectId, ra, decl, z, ebv, peakMag, peakBand, absMag))


170019696300523640: ra=60.15 decl=-49.05 z=0.004 E(B-V)=0.01 peakMax=22.71 peakBand=z, absMag=-8.20
170019696304193611: ra=59.43 decl=-48.91 z=0.004 E(B-V)=0.01 peakMax=22.89 peakBand=r, absMag=-8.16
170019696393848492: ra=65.27 decl=-48.30 z=0.016 E(B-V)=0.01 peakMax=23.11 peakBand=g, absMag=-11.04
170019696464101449: ra=62.09 decl=-47.91 z=0.004 E(B-V)=0.01 peakMax=23.42 peakBand=r, absMag=-7.78
170019716255973493: ra=149.34 decl=0.96 z=0.192 E(B-V)=0.03 peakMax=22.04 peakBand=z, absMag=-17.82
170019716270129241: ra=150.34 decl=0.81 z=0.675 E(B-V)=0.02 peakMax=25.17 peakBand=i, absMag=-17.88
170019716272226349: ra=149.90 decl=0.86 z=0.759 E(B-V)=0.02 peakMax=24.84 peakBand=i, absMag=-18.52
170019716273274936: ra=149.68 decl=1.11 z=0.471 E(B-V)=0.03 peakMax=23.19 peakBand=z, absMag=-18.91
170019716279566437: ra=149.44 decl=1.91 z=0.411 E(B-V)=0.02 peakMax=23.58 peakBand=r, absMag=-18.18
170019716291100725: ra=150.93 decl=0.64 z=0.730 E(B-V)=0.03 peakMax=25.30 peakBand=g, absMag=-17.96